# Workshop 4

This is a Jupyter Notebook. You can add cells that contain code in Python (or other programming languages) or cells that contain text such as this one.

In [2]:
# This is a Python cell that loads data from the main file

%pip install pandas scikit-learn matplotlib


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:

import pandas as pd

# Load the IMD data
IMD_df = pd.read_csv("DATA.csv")

key = "LSOA code (2021)"
lad = "Local Authority District code (2024)"
score_cols = IMD_df.columns[IMD_df.columns.str.contains("Score")]
IMD_df_scores = df[[key, lad, *score_cols]]

Benefits_df = pd.read_csv("Benefit claimants.csv", skiprows=list(range(8)) + [9],)

Benefits_df = Benefits_df.drop(columns=Benefits_df.columns[0])
Benefits_df["August 2026"] = pd.to_numeric(
    Benefits_df["August 2026"].str.replace(",", "", regex=False),
    errors="coerce",
)
Benefits_df = Benefits_df.rename(columns={
    "mnemonic": "LSOA code (2021)",
    "August 2026": "proportion benefit claimants",
})

# print(Benefits_df.head())
# print(Benefits_df.info())

Ethnicity_df = pd.read_csv("Ethnicity_data.csv", skiprows=list(range(7)) + [8],)

Ethnicity_df = Ethnicity_df.rename(columns={
    "2021 super output area - lower layer": "LSOA code (2021)",
    "Total: All usual residents": "total_residents",
    "Asian, Asian British or Asian Welsh": "asian",
    "Black, Black British, Black Welsh, Caribbean or African": "black",
    "Mixed or Multiple ethnic groups": "mixed",
    "White": "white",
    "Other ethnic group": "other_ethnic",
})

Ethnicity_df["total_residents"] = pd.to_numeric(
    Ethnicity_df["total_residents"].str.replace(",", "", regex=False),
    errors="coerce",
)
# Check NaN rows and drop 
# print(Ethnicity_df[Ethnicity_df.isna().any(axis=1)])
Ethnicity_df = Ethnicity_df.dropna(subset=["LSOA code (2021)", "total_residents"])

# Convert to proportions
group_cols = ["asian", "black", "mixed", "white", "other_ethnic"]
for c in group_cols:
    Ethnicity_df[f"{c}_prop"] = Ethnicity_df[c] / Ethnicity_df["total_residents"]

# Proportion of residents not in the White group
Ethnicity_df["minority_prop"] = 1 - Ethnicity_df["white_prop"]

# Adjust string used as key to remove description
Ethnicity_df["LSOA code (2021)"] = (
    Ethnicity_df["LSOA code (2021)"]
    .str.split(":").str[0]
    .str.strip()
)
# checks... these lines helped me look at the output. 
# Ethnicity_df["minority_prop"].describe().round(2)

# Ethnicity_df.info()

NameError: name 'df' is not defined

The next cell joins the data together into one large table and prints out a summary of what is in the combined table. 

In [3]:

combined_df = (
    IMD_df_scores
    .merge(Benefits_df, how="left", on="LSOA code (2021)")
    .merge(Ethnicity_df, how="left", on="LSOA code (2021)")
)

print(combined_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 31 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   LSOA code (2021)                                                  33755 non-null  object 
 1   Local Authority District code (2024)                              33755 non-null  str    
 2   Index of Multiple Deprivation (IMD) Score                         33755 non-null  float64
 3   Income Score (rate)                                               33755 non-null  float64
 4   Employment Score (rate)                                           33755 non-null  float64
 5   Education, Skills and Training Score                              33755 non-null  float64
 6   Health Deprivation and Disability Score                           33755 non-null  float64
 7   Crime Score                               

## Task 1 - Feature Leakage. 

We are going to cluster the domains within the IMD data. The next cell of code labels the domains we will focus on. 

1. Run the code in the block below. 
2. Ask the LLM how the IMD is calculated. 
2. Ask your preferred LLM to write you some code in Python to examine how these domains are related. You could look at the correlations between each domain or whether the IMD or proportion of benefit claimants can be predicted using the domains. What data goes into the Income and Employment domains?
3. Ask your preferred LLM to cluster the domain scores using the K-means algorithm using K = 4. What are the typical features of LSOAs in each group? 
4. Add the IMD score to the domain scores and repeat the clustering with K = 4. Is this clustering "better"? How does your LLM suggest you measure better? How many of the original cluster assignments change? (Ask your LLM for the adjusted Rand index.)
5. Did the code rescale the variables? Why does it matter?


In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import adjusted_rand_score

prefixes = {
    "imd": "Index of Multiple Deprivation (IMD) Score",
    "income": "Income Score",
    "employment": "Employment Score",
    "education": "Education, Skills",
    "health": "Health Deprivation",
    "crime": "Crime Score",
    "barriers": "Barriers to Housing",
    "living_env": "Living Environment Score",
}
cols = {k: next(c for c in combined_df.columns if c.startswith(v)) for k, v in prefixes.items()}

d = combined_df[list(cols.values()) + ["proportion benefit claimants"]].copy()
d = d.rename(columns={v: k for k, v in cols.items()})
d = d.rename(columns={"proportion benefit claimants": "benefits"}).dropna()

domains = ["income", "employment", "education", "health", "crime", "barriers", "living_env"]

There is more information on the slides for this Workshop about what went wrong here!

## TASK 2 - Cross Validation

As discussed in Workshop 2 - Machine Learning/AI algorithms typically require a train/test split to validate performance on unseen data. In the previous task we did not split the data. 

A graphic about cross-validation can be found on the slides. Validation on unseen data can be a good way of spotting **some** types data leakage. It does this by hiding part of the data, training on the rest, and checking the predictions against the hidden part. The usual approach assigns rows to the hidden part at random. 

1. Ask your preferred LLM: "What does it mean to say that data are *spatially autocorrelated*? Why might this matter for LSOAs?" Write down the answer in one or two sentences of your own.
2. Ask your LLM to write Python code that predicts the proportion of benefit claimants from the other domain scores using a random forest, and evaluates it with standard 5-fold cross-validation. What score does it report? 
3. Now ask your LLM to repeat this, but so that *whole local authority districts* are held out together in each fold (it may suggest `GroupKFold`). How does the score change?

In [5]:
# A code cell for you to paste things in :)

## Task 3 - Proxies for Ethnicity

The domain scores in the IMD do not include ethnicity. In this task we ask whether that means a model built from them is "ethnicity-blind".

1. Ask your preferred LLM to write code that clusters the LSOAs using the seven domain scores (K = 4, standardised), then shows the ethnic composition of each cluster. Do the clusters differ? Describe the pattern in a sentence or two.
2. Ask your LLM to write code that tries to *predict* the minority ethnic proportion of an LSOA using the domain scores alone. How well does it do? (Think about how you want to validate it. What did you learn in Task 2?)
3. Now remove the Income and Employment domains, and repeat step 3. Does the model still predict ethnicity? What does this tell you about removing "sensitive" features from a dataset?
4. Ask your LLM: "What is the *ecological fallacy*?" Then answer: if a cluster has a high proportion of ethnic minority residents, what can you validly say about an individual who lives there?
